In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [ ]:
file_path = "Pakistan_Gold_Daily_Regression_2025_to_2026-09-01.xlsx"
df = pd.read_excel(file_path, sheet_name="Daily_Regression")

display(df.head())
df.info()

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['Quarter'] = df['Date'].dt.quarter

df['Yesterday_Price'] = df['24K_PKR_per_Tola'].shift(1)
df['Price_2_Days_Ago'] = df['24K_PKR_per_Tola'].shift(2)
df['Price_7_Days_Ago'] = df['24K_PKR_per_Tola'].shift(7)
df['7_Day_Moving_Avg'] = df['24K_PKR_per_Tola'].shift(1).rolling(window=7).mean()

df_clean = df.dropna().reset_index(drop=True)
display(df_clean.head())

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df_clean['Date'], df_clean['24K_PKR_per_Tola'], color='goldenrod', label='Actual Price')
plt.plot(df_clean['Date'], df_clean['7_Day_Moving_Avg'], color='navy', linestyle='--', label='7-Day Moving Avg')
plt.title('Pakistan 24K Gold Price Trend (2025 - 2026)')
plt.xlabel('Date')
plt.ylabel('Price in PKR (per Tola)')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

In [ ]:
target = '24K_PKR_per_Tola'
columns_to_drop = [
    'Date',
    'Day_of_Week',
    '24K_Daily_Change_PKR',
    '24K_Daily_Change_Pct',
    'Fill_Basis_Date',
    target
]

X = df_clean.drop(columns=columns_to_drop)
y = df_clean[target]

split_point = int(len(df_clean) * 0.80)
X_train, X_test = X.iloc[:split_point], X.iloc[split_point:]
y_train, y_test = y.iloc[:split_point], y.iloc[split_point:]

print(f"Train: {len(X_train)} | Test: {len(X_test)}")

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.1),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    results.append({
        "Model": name,
        "R2 Score": round(r2_score(y_test, y_pred), 4),
        "MAE (PKR)": round(mean_absolute_error(y_test, y_pred), 2),
        "RMSE (PKR)": round(np.sqrt(mean_squared_error(y_test, y_pred)), 2)
    })

results_df = pd.DataFrame(results).sort_values(by="R2 Score", ascending=False)
display(results_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=results_df, x="R2 Score", y="Model", ax=axes[0], palette="Blues_r")
axes[0].set_title("R2 Score (Higher is Better)")
axes[0].set_xlim(0, 1.0)

sns.barplot(data=results_df, x="MAE (PKR)", y="Model", ax=axes[1], palette="Oranges_r")
axes[1].set_title("MAE in PKR (Lower is Better)")

plt.tight_layout()
plt.show()

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]
best_predictions = best_model.predict(X_test_scaled)

test_dates = df_clean['Date'].iloc[split_point:]

plt.figure(figsize=(13, 6))
plt.plot(test_dates, y_test.values, label="Actual Price", color="black", linewidth=2)
plt.plot(test_dates, best_predictions, label=f"Predicted ({best_model_name})", color="crimson", linestyle="--", linewidth=2)
plt.title(f"Gold Price Forecast vs Actual ({best_model_name})")
plt.xlabel("Date")
plt.ylabel("Price (PKR per Tola)")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.6)
plt.show()

In [ ]:
future_data = df_clean.copy()
future_predictions = []

for i in range(30):
    last_date = future_data['Date'].iloc[-1]
    next_date = last_date + pd.Timedelta(days=1)
    
    new_features = pd.DataFrame([{
        'Day_of_Week_Num': next_date.dayofweek + 1,
        'Is_Observed': 0 if next_date.dayofweek == 6 else 1,
        'Year': next_date.year,
        'Month': next_date.month,
        'Day': next_date.day,
        'Quarter': next_date.quarter,
        'Yesterday_Price': future_data['24K_PKR_per_Tola'].iloc[-1],
        'Price_2_Days_Ago': future_data['24K_PKR_per_Tola'].iloc[-2],
        'Price_7_Days_Ago': future_data['24K_PKR_per_Tola'].iloc[-7],
        '7_Day_Moving_Avg': future_data['24K_PKR_per_Tola'].iloc[-7:].mean()
    }])[X.columns]

    new_features_scaled = scaler.transform(new_features)
    predicted_price = best_model.predict(new_features_scaled)[0]

    future_predictions.append({
        'Date': next_date,
        'Day': next_date.strftime('%A'),
        'Predicted_Price_PKR': round(predicted_price, 2)
    })

    new_row = new_features.copy()
    new_row['Date'] = next_date
    new_row['24K_PKR_per_Tola'] = predicted_price
    future_data = pd.concat([future_data, new_row], ignore_index=True)

forecast_df = pd.DataFrame(future_predictions)
display(forecast_df)

In [ ]:
plt.figure(figsize=(14, 6))
recent_history = df_clean.tail(60)

plt.plot(recent_history['Date'], recent_history['24K_PKR_per_Tola'], color='black', lw=2, label='Recent Actual Price')
plt.plot(forecast_df['Date'], forecast_df['Predicted_Price_PKR'], color='crimson', linestyle='--', lw=2, marker='o', markersize=4, label=f'30-Day Forecast ({best_model_name})')

plt.title("Pakistan 24K Gold Price - Next 30 Days Forecast")
plt.xlabel("Date")
plt.ylabel("Price (PKR per Tola)")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.6)
plt.show()